### Fine-tuning DEBERTA

In [1]:
import os
# Allow PyTorch to fall back to CPU for MPS-unsupported ops
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

import torch
import transformers
import datasets
import sklearn

print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"sklearn:      {sklearn.__version__}")

# Check MPS availability
print(f"\nMPS available:    {torch.backends.mps.is_available()}")
print(f"MPS built:        {torch.backends.mps.is_built()}")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Selected device:  {device}")

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


torch:        2.8.0
transformers: 4.57.6
datasets:     4.5.0
sklearn:      1.6.1

MPS available:    True
MPS built:        True
Selected device:  mps


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "microsoft/deberta-v3-base"
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Vocab size:       {tokenizer.vocab_size}")

sample = "I auditioned for the surf team and the coach avoided me."
tokens = tokenizer.tokenize(sample)
ids = tokenizer.encode(sample)
print(f"\nSample text:   {sample}")
print(f"Tokens ({len(tokens)}): {tokens}")
print(f"IDs ({len(ids)}):    {ids}")

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Tokenizer loaded: DebertaV2TokenizerFast
Vocab size:       128000

Sample text:   I auditioned for the surf team and the coach avoided me.
Tokens (12): ['▁I', '▁auditioned', '▁for', '▁the', '▁surf', '▁team', '▁and', '▁the', '▁coach', '▁avoided', '▁me', '.']
IDs (14):    [1, 273, 62808, 270, 262, 10873, 511, 263, 262, 2305, 8549, 351, 260, 2]


- 128,000-token vocabulary
- Token IDs 1 at the start and 2 at the end are the special tokens added automatically. ID 1 is [CLS], ID 2 is [SEP].
```
[CLS]  ▁I  ▁auditioned  ▁for  ▁the  ▁surf  ▁team  ▁and  ▁the  ▁coach  ▁avoided  ▁me  .  [SEP]
    1   273    62808       270   262   10873   511    263   262   2305    8549     351 260   2
```
This is the input format DeBERTa expects, and when the model runs, it will produce 14 output vectors; one per token, and the classifier will read only the first one (the [CLS] vector at position 0).


In [3]:
# Load the model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=11,  # 11 for TherapistQA; 10 for PatternReframe
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model on device:      {next(model.parameters()).device}")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters:     184,430,603
Trainable parameters: 184,430,603
Model on device:      mps:0


This downloads ~750 MB on first run. Classification head weights are newly initialised; that is the fresh head being attached.

Hugging Face is listing exactly which weights in the model are random versus pretrained. Four weight tensors are random:
- `classifier.weight` and `classifier.bias`: the classification head proper. The (768 → 11) matrix and its bias vector. Together about 8,459 random numbers.
- `pooler.dense.weight` and `pooler.dense.bias`: a small intermediate layer DeBERTa adds between the [CLS] vector and the classifier. It applies one more transformation (a 768-dim linear layer + tanh activation) before the classification head reads the result. About 590,000 random numbers.
```
[CLS] vector (768d, contextual, from DeBERTa)
        ↓
      pooler   - 768 → 768 linear + tanh   (RANDOM, ~590K params)
        ↓
    classifier - 768 → 11 linear           (RANDOM, ~8.5K params)
        ↓
    11 numbers - softmax → probabilities
```


Check how many records exceed context window (512 tokens for DEBERTA):

In [4]:
import pandas as pd

df = pd.read_csv("./processed/therapistqa.csv")

# Tokenise every record without truncation, just to count
token_counts = df["patient_question"].astype(str).apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True))
)

print(f"Token count statistics:")
print(f"  Mean:   {token_counts.mean():.0f}")
print(f"  Median: {token_counts.median():.0f}")
print(f"  Min:    {token_counts.min()}")
print(f"  Max:    {token_counts.max()}")
print(f"  95th %: {token_counts.quantile(0.95):.0f}")
print(f"  99th %: {token_counts.quantile(0.99):.0f}")
print(f"\nRecords exceeding 512 tokens: {(token_counts > 512).sum()} / {len(df)} "
      f"({100 * (token_counts > 512).mean():.1f}%)")
print(f"Records exceeding 256 tokens: {(token_counts > 256).sum()} / {len(df)} "
      f"({100 * (token_counts > 256).mean():.1f}%)")

Token count statistics:
  Mean:   200
  Median: 152
  Min:    46
  Max:    1339
  95th %: 470
  99th %: 574

Records exceeding 512 tokens: 51 / 2526 (2.0%)
Records exceeding 256 tokens: 660 / 2526 (26.1%)


**2% of records exceeded the 512-token window**
- 98% of records fit fully, no truncation at all
- Of the 2% that get truncated, most lose only 50–60 tokens off the end (since the 99th percentile is 574). That is one or two sentences of trailing content.
- The published baseline by Shreevastava and Foltz (2021) on the same TherapistQA dataset used 512 with truncation. Matching their setup keeps the comparison fair.

### Stratified train/val/test split for TherapistQA

In [5]:
from sklearn.model_selection import train_test_split
from pathlib import Path

# 1. First split: 15% test
df_trainval, df_test = train_test_split(
    df,
    test_size=0.15,
    stratify=df["dominant_distortion"],
    random_state=42,  # 42 is "the answer to the ultimate question of life, the universe, and everything" :)
)

# 2. Second split: 15% val 70% train
df_train, df_val = train_test_split(
    df_trainval,
    test_size=0.15 / 0.85,
    stratify=df_trainval["dominant_distortion"],
    random_state=42,
)

print(f"Train: {len(df_train):>5} ({100 * len(df_train) / len(df):.1f}%)")
print(f"Val:   {len(df_val):>5} ({100 * len(df_val) / len(df):.1f}%)")
print(f"Test:  {len(df_test):>5} ({100 * len(df_test) / len(df):.1f}%)")
print(f"Total: {len(df_train) + len(df_val) + len(df_test):>5}")

print("\nClass proportions across splits (should be similar):")
proportions = pd.DataFrame({
    "train": df_train["dominant_distortion"].value_counts(normalize=True),
    "val":   df_val["dominant_distortion"].value_counts(normalize=True),
    "test":  df_test["dominant_distortion"].value_counts(normalize=True),
}).round(3)
print(proportions.to_string())

# 3. Save the three splits
output_dir = Path("processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_train.to_csv(output_dir / "therapistqa_train.csv", index=False)
df_val.to_csv(output_dir / "therapistqa_val.csv", index=False)
df_test.to_csv(output_dir / "therapistqa_test.csv", index=False)

print(f"\nSaved to {output_dir}/")

Train:  1768 (70.0%)
Val:     379 (15.0%)
Test:    379 (15.0%)
Total:  2526

Class proportions across splits (should be similar):
                         train    val   test
dominant_distortion                         
All-or-nothing thinking  0.039  0.040  0.040
Emotional Reasoning      0.053  0.053  0.053
Fortune-telling          0.057  0.055  0.055
Labeling                 0.065  0.066  0.066
Magnification            0.077  0.077  0.077
Mental filter            0.049  0.047  0.047
Mind Reading             0.094  0.095  0.095
No Distortion            0.369  0.369  0.369
Overgeneralization       0.094  0.095  0.095
Personalization          0.060  0.061  0.061
Should statements        0.042  0.042  0.042

Saved to processed/


In [3]:
import pandas as pd
from datasets import Dataset, DatasetDict

# 1. Load the three CSVs as pandas (if using premade splits), then convert to HF Datasets
df_train = pd.read_csv("./processed/therapistqa_train.csv")
df_val   = pd.read_csv("./processed/therapistqa_val.csv")
df_test  = pd.read_csv("./processed/therapistqa_test.csv")

# 2. Build label dictionaries from the training set's classes (sorted for determinism)
classes = sorted(df_train["dominant_distortion"].unique())
label2id = {label: idx for idx, label in enumerate(classes)}
id2label = {idx: label for label, idx in label2id.items()}

print("Label mapping:")
for idx, label in id2label.items():
    print(f"  {idx:>2}  →  {label}")

# 3. Build HuggingFace Datasets, keeping only the columns we need + an integer label column
def make_dataset(df):
    return Dataset.from_pandas(
        df[["patient_question", "dominant_distortion"]]
            .rename(columns={"patient_question": "text"})
            .assign(label=lambda d: d["dominant_distortion"].map(label2id))
            [["text", "label"]],
        preserve_index=False,
    )

raw_datasets = DatasetDict({
    "train":      make_dataset(df_train),
    "validation": make_dataset(df_val),
    "test":       make_dataset(df_test),
})

print("\nDataset sizes:")
for split, ds in raw_datasets.items():
    print(f"  {split:<11}  {len(ds):>5} records")

# 4. Tokenise: applied to all splits at once via .map()
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        # Note: no padding here - the data collator will pad per-batch later, more efficient
    )

tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],   # drop the raw text after tokenisation
)

print("\nA tokenised example:")
example = tokenized_datasets["train"][0]
print(f"  label:          {example['label']} ({id2label[example['label']]})")
print(f"  num tokens:     {len(example['input_ids'])}")
print(f"  first 15 IDs:   {example['input_ids'][:15]}")
print(f"  attention mask: {example['attention_mask'][:15]}")

Label mapping:
   0  →  All-or-nothing thinking
   1  →  Emotional Reasoning
   2  →  Fortune-telling
   3  →  Labeling
   4  →  Magnification
   5  →  Mental filter
   6  →  Mind Reading
   7  →  No Distortion
   8  →  Overgeneralization
   9  →  Personalization
  10  →  Should statements

Dataset sizes:
  train         1768 records
  validation     379 records
  test           379 records


Map:   0%|          | 0/1768 [00:00<?, ? examples/s]

Map:   0%|          | 0/379 [00:00<?, ? examples/s]

Map:   0%|          | 0/379 [00:00<?, ? examples/s]


A tokenised example:
  label:          7 (No Distortion)
  num tokens:     92
  first 15 IDs:   [1, 1007, 266, 7128, 267, 262, 543, 260, 430, 260, 294, 5365, 261, 273, 276]
  attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


**Padding deferred.** `tokenize_function` does not pad.
- Padding can be done here to a fixed length (say 512), but every record then becomes a 512-long array regardless of its actual length, wasting compute on padding tokens.
- The cleaner approach is dynamic padding: the `Trainer` will use a `DataCollatorWithPadding` later that pads each batch to its longest member, which on average is far shorter than 512. Faster training, no information lost.

### Load the model with label dicts

In [4]:
# Reload the model with the proper label dictionaries
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

print(f"Model num_labels: {model.config.num_labels}")
print(f"Model id2label:   {model.config.id2label}")
print(f"On device:        {next(model.parameters()).device}")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model num_labels: 11
Model id2label:   {0: 'All-or-nothing thinking', 1: 'Emotional Reasoning', 2: 'Fortune-telling', 3: 'Labeling', 4: 'Magnification', 5: 'Mental filter', 6: 'Mind Reading', 7: 'No Distortion', 8: 'Overgeneralization', 9: 'Personalization', 10: 'Should statements'}
On device:        mps:0


### Configure the Trainer

- `weight_decay=0.01`: lightly regularises the weights to prevent overfitting;
- `warmup_ratio=0.1`: ramps the learning rate from 0 up to 2e-5 over the first 10% of training steps, which stabilises the early epochs (transformers can be brittle if hit with full learning rate from step 1. After warmup, the learning rate decays linearly back down to 0 over the rest of training (the default scheduler).
    -  In the very first training batch, the model produces near-random predictions. The loss is enormous. The gradients computed from that loss are also enormous; *everything is wrong, change everything*.
    - If the full learning rate 2e-5 is applied to those huge first-batch gradients the pretrained DeBERTa weights get yanked far from their good positions. The model forgets what it knew (catastrophic forgetting).
    - The first step uses learning rate 0 (no update at all). The next step uses a tiny rate. By the time the rate reaches the full 2e-5 around step 56, the head has already learned the rough shape of the task using small updates, the gradients have settled into normal magnitudes and no good weights get damaged before the random head has a chance to align with the task.
    - The decay phase does the opposite: as training progresses, the learning rate shrinks so the model makes smaller and smaller adjustments. This helps it settle into a good minimum rather than bouncing around it. Late epochs do *fine polishing* rather than *big restructuring*.
- `save_total_limit=2`: without this, every epoch saves a full checkpoint (~750 MB each), and the disk fills up fast. Keeping the 2 most recent.
- `load_best_model_at_end=True`: ensures the best one is kept regardless.
- `fp16=False`, `bf16=False`: explicit. Mixed precision is not supported on MPS in current PyTorch.
- `per_device_train_batch_size=16` and `per_device_eval_batch_size=32`:
    - A batch is the **number of examples the model processes together** before computing gradients and updating its weights. The phrase "per device" matters when training across multiple GPUs - irrelevant here since the M4 is one device, so this is just "the batch size."
    - Larger batches smooth out the gradient (less noisy weight updates) and run more efficiently on a GPU (better parallelism).
    - Smaller batches use less GPU memory and introduce a small amount of noise that can actually help generalisation.
    - `per_device_eval_batch_size=32`: evaluation does not need gradients, so memory is freer and the batch can be doubled - makes the eval pass at the end of each epoch faster.
- `num_train_epochs=5`:
    - An epoch is **one complete pass through the training set**.
    - With 1,768 training records and batch size 16, one epoch is 1,768 / 16 ~ 111 batches.
    - Five epochs means the model sees the entire training set five times - about 555 batches total.
    - Empirically, fine-tuning BERT-family models tends to converge in 3–5 epochs and then start overfitting (training F1 keeps rising while validation F1 plateaus or drops). The plan picks 5 as a safe upper bound, and the early stopping callback (more on that below) cuts it short if validation F1 stops improving - so in practice training often finishes after epoch 3 or 4.
- `EarlyStoppingCallback(early_stopping_patience=2)`: patience 2 means: *if validation f1_weighted does not improve for 2 consecutive epochs, stop training.*
- `seed=42`: Fixes the random number generator inside the Trainer itself. This affects any stochastic behaviour in the training loop.

In [5]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

# 1. Data collator: handles dynamic padding per batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. Metrics function: called by Trainer at the end of each epoch
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy":    accuracy_score(labels, predictions),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
        "f1_macro":    f1_score(labels, predictions, average="macro"),
    }

# 3. Training arguments
training_args = TrainingArguments(
    output_dir="models/deberta_therapistqa_seed42",

    # Optimiser
    learning_rate=2e-5,
    weight_decay=0.01,

    # Schedule
    num_train_epochs=5,
    per_device_train_batch_size=8,           # reduced from 16
    per_device_eval_batch_size=8,            # reduced from 32
    gradient_accumulation_steps=2,           # effective batch size still 16
    warmup_ratio=0.1,

    # Evaluation and checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,                  # keep only the 2 most recent checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,

    # Logging
    logging_strategy="epoch",
    report_to="none",                    # no Weights & Biases / TensorBoard

    # Reproducibility
    seed=42,

    # MPS-specific: disable fp16, leave bf16 off too
    fp16=False,
    bf16=False,
)

# 4. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Trainer configured.")
print(f"  Output dir:        {training_args.output_dir}")
print(f"  Epochs:            {training_args.num_train_epochs}")
print(f"  Train batch size:  {training_args.per_device_train_batch_size}")
print(f"  Learning rate:     {training_args.learning_rate}")
print(f"  Best metric:       {training_args.metric_for_best_model}")
print(f"  Early stopping:    patience 2")

Trainer configured.
  Output dir:        models/deberta_therapistqa_seed42
  Epochs:            5
  Train batch size:  8
  Learning rate:     2e-05
  Best metric:       f1_weighted
  Early stopping:    patience 2


### Train the model

In [ ]:
import time

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

print(f"\n{'='*60}")
print(f"Training finished in {elapsed/60:.1f} minutes")
print(f"{'='*60}")
print(f"\nFinal training metrics:")
for k, v in train_result.metrics.items():
    print(f"  {k}: {v}")